# Tornado Experiment — Master Runner (Morocco)

Executes the full analysis pipeline without visualizations:

1. **Load run data** — attribute tables + wide-format simulation output
2. **Post-processing** — intertemporal decomposition / rescaling → `decomposed_ssp_output_tornado.csv`
3. **Cost-benefits** — system + technical costs → `cost_benefits_data_tornado.csv`
4. **MAC analysis** — marginal abatement costs → `marginal_abatement_costs_tornado.csv`
5. **Tableau export** — tornado plot data → `tableau/data/tableau_tornado.csv`

**Tornado design.** Every strategy adds exactly one transformation on top of
`TX:BASE` (`TORNADO_BASE:TX:<SECTOR>:<NAME>`), so `BASE` is the reference for
both emissions and costs.

All configuration lives in `scripts/config.py` — country, reference year,
inventory files and the run to analyze are read from
`notebooks/config_files/config.yaml` plus a few explicit constants, so nothing
here is hard-coded per country.
Shared pipeline modules (`data_loading`, `postprocessing`,
`cost_benefits_pipeline`) come from `../shared_scripts/` — the Morocco versions
used by `workflow/morocco_manager_wb.ipynb`.


## 0 · Environment setup

In [10]:
import os
import sys
import pathlib
import logging
import warnings

%load_ext autoreload
%autoreload 2

warnings.filterwarnings("ignore")

# ── Paths ─────────────────────────────────────────────────────────────────────
RUNNER_DIR  = pathlib.Path(os.getcwd()).resolve()
SCRIPTS_DIR = RUNNER_DIR / "scripts"
SHARED_DIR  = RUNNER_DIR.parent / "shared_scripts"

assert SCRIPTS_DIR.exists(), f"scripts/ not found at {SCRIPTS_DIR}"
assert SHARED_DIR.exists(),  f"shared_scripts/ not found at {SHARED_DIR}"

for p in (SCRIPTS_DIR, SHARED_DIR):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

# ── Config ────────────────────────────────────────────────────────────────────
import config as cfg

# Project root (needed for ssp_modeling.* imports inside pipeline scripts)
if str(cfg.PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(cfg.PROJECT_DIR))

# notebooks/ dir (needed for utils.logger_utils)
if str(cfg.NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(cfg.NOTEBOOKS_DIR))

from utils.logger_utils import setup_clean_logger, mute_external_loggers

logger = setup_clean_logger("tornado", logging.INFO)
mute_external_loggers(["sisepuede"])
logger.info("Environment ready.")
logger.info(f"Project dir   : {cfg.PROJECT_DIR}")
logger.info(f"Run ID        : {cfg.RUN_ID}")
logger.info(f"Run output    : {cfg.RUN_ID_OUTPUT_DIR}")
logger.info(f"Region / ISO  : {cfg.REGION} / {cfg.ISO_CODE3}")
logger.info(f"Years         : {cfg.YEAR_START}-{cfg.YEAR_END} (ref {cfg.YEAR_REF})")
logger.info(f"Targets       : {cfg.TARGETS_PATH.name}")
logger.info(f"Inventory     : {cfg.INVENT_HISTORIC_PATH.name}")
logger.info(f"Tableau dir   : {cfg.TABLEAU_DIR}")
logger.info(f"Primary IDs   : {len(cfg.PRIMARY_IDS_FILTER)}")

# Fail fast if the run or the reference data are missing
for _label, _path in [
    ("targets",   cfg.TARGETS_PATH),
    ("inventory", cfg.INVENT_HISTORIC_PATH),
    ("cb config", cfg.CB_CONFIG_PATH),
    ("run dir",   cfg.RUN_ID_OUTPUT_DIR),
]:
    assert _path.exists(), f"{_label} not found: {_path}"

2026-09-17 17:24:09,315 - INFO - Environment ready.
2026-09-17 17:24:09,316 - INFO - Project dir   : /Users/fabianfuentes/git/ssp_morocco
2026-09-17 17:24:09,316 - INFO - Run ID        : sisepuede_results_sisepuede_run_2026-09-17T16;54;17.970763
2026-09-17 17:24:09,316 - INFO - Run output    : /Users/fabianfuentes/git/ssp_morocco/ssp_modeling/ssp_run_output/sisepuede_results_sisepuede_run_2026-09-17T16;54;17.970763
2026-09-17 17:24:09,317 - INFO - Region / ISO  : morocco / MAR
2026-09-17 17:24:09,317 - INFO - Years         : 2015-2050 (ref 2018)
2026-09-17 17:24:09,317 - INFO - Targets       : emission_targets_mar_2018.csv
2026-09-17 17:24:09,318 - INFO - Inventory     : invent_historic_mar.csv
2026-09-17 17:24:09,318 - INFO - Tableau dir   : /Users/fabianfuentes/git/ssp_morocco/ssp_modeling/tableau/data
2026-09-17 17:24:09,318 - INFO - Primary IDs   : 44


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1 · Load run data

In [11]:
from data_loading import load_attribute_tables, parse_strategy_metadata, load_wide_export

# Attribute tables
att_primary, att_strategy = load_attribute_tables(cfg.RUN_ID_OUTPUT_DIR)
att_strategy = parse_strategy_metadata(att_strategy)

logger.info(f"att_primary  : {att_primary.shape}")
logger.info(f"att_strategy : {att_strategy.shape}")

# Wide-format simulation output
df_export = load_wide_export(cfg.RUN_ID_OUTPUT_DIR, cfg.PRIMARY_IDS_FILTER)
logger.info(f"df_export    : {df_export.shape}")

2026-09-17 17:24:09,337 - INFO - att_primary  : (44, 4)
2026-09-17 17:24:09,337 - INFO - att_strategy : (158, 10)
2026-09-17 17:24:09,660 - INFO - df_export    : (1584, 4119)


## 2 · Post-processing — intertemporal decomposition

In [12]:
from postprocessing import run_decomposition

df_decomposed = run_decomposition(
    df_export    = df_export,
    project_dir  = cfg.PROJECT_DIR,
    targets_path = cfg.TARGETS_PATH,
    iso_code3    = cfg.ISO_CODE3,
    year_ref     = cfg.YEAR_REF,
    region       = cfg.REGION,
    output_path  = cfg.OUTPUT_DECOMPOSED,
    initial_conditions_id = cfg.INITIAL_CONDITIONS_ID,   # BASE → primary_id 0
)
logger.info(f"df_decomposed : {df_decomposed.shape}  → {cfg.OUTPUT_DECOMPOSED}")

Changed 1452 zero(s) in: emission_co2e_ch4_entc_fuel_mining_and_extraction_me_coal (across all years for primary_ids [np.int64(0), np.int64(73073), np.int64(74074), np.int64(75075), np.int64(76076), np.int64(77077), np.int64(78078), np.int64(79079), np.int64(80080), np.int64(81081), np.int64(82082), np.int64(83083), np.int64(84084), np.int64(85085), np.int64(86086), np.int64(87087), np.int64(88088), np.int64(89089), np.int64(90090), np.int64(91091), np.int64(92092), np.int64(93093), np.int64(94094), np.int64(95095), np.int64(96096), np.int64(97097), np.int64(98098), np.int64(99099), np.int64(100100), np.int64(101101), np.int64(102102), np.int64(103103), np.int64(104104), np.int64(105105), np.int64(106106), np.int64(107107), np.int64(108108), np.int64(109109), np.int64(110110), np.int64(111111), np.int64(112112), np.int64(113113), np.int64(114114), np.int64(115115)])
Changed 1452 zero(s) in: emission_co2e_ch4_entc_fuel_mining_and_extraction_me_crude (across all years for primary_ids [np

2026-09-17 17:24:16,201 - INFO - df_decomposed : (1452, 4119)  → /Users/fabianfuentes/git/ssp_morocco/ssp_modeling/ssp_run_output/sisepuede_results_sisepuede_run_2026-09-17T16;54;17.970763/decomposed_ssp_output_tornado.csv


Decomposed output written to: /Users/fabianfuentes/git/ssp_morocco/ssp_modeling/ssp_run_output/sisepuede_results_sisepuede_run_2026-09-17T16;54;17.970763/decomposed_ssp_output_tornado.csv
Done: morocco


## 3 · Cost-benefits

In [13]:
from cost_benefits_pipeline import run_cost_benefits

cb_data = run_cost_benefits(
    df_decomposed      = df_decomposed,
    att_primary        = att_primary,
    att_strategy       = att_strategy,
    cb_config_path     = cfg.CB_CONFIG_PATH,
    run_output_dir     = cfg.RUN_ID_OUTPUT_DIR,
    project_dir        = cfg.PROJECT_DIR,
    strategy_code_base = cfg.STRATEGY_CODE_BASE,
    output_path        = cfg.OUTPUT_CB_DATA,
)
logger.info(f"cb_data : {cb_data.shape}  → {cfg.OUTPUT_CB_DATA}")

Loading configuration from Excel file (fast path)
Database updated

************************************
*Strategy : PFLO:LEDS (0/43)
************************************

---------Costs for: cb:agrc:crop_value:crops_produced:bevs_and_spices.
The variable is evaluated in System Cost
---------Costs for: cb:agrc:crop_value:crops_produced:cereals.
The variable is evaluated in System Cost
---------Costs for: cb:agrc:crop_value:crops_produced:fibers.
The variable is evaluated in System Cost
---------Costs for: cb:agrc:crop_value:crops_produced:fruits.
The variable is evaluated in System Cost
---------Costs for: cb:agrc:crop_value:crops_produced:herbs.
The variable is evaluated in System Cost
---------Costs for: cb:agrc:crop_value:crops_produced:nuts.
The variable is evaluated in System Cost
---------Costs for: cb:agrc:crop_value:crops_produced:other_annual.
The variable is evaluated in System Cost
---------Costs for: cb:agrc:crop_value:crops_produced:other_woody_perennial.
The variable is e

2026-09-17 17:24:46,020 - INFO - cb_data : (305978, 21)  → /Users/fabianfuentes/git/ssp_morocco/ssp_modeling/ssp_run_output/sisepuede_results_sisepuede_run_2026-09-17T16;54;17.970763/cost_benefits_data_tornado.csv


[cb_pipeline] Saved 305,978 rows → /Users/fabianfuentes/git/ssp_morocco/ssp_modeling/ssp_run_output/sisepuede_results_sisepuede_run_2026-09-17T16;54;17.970763/cost_benefits_data_tornado.csv


## 4 · Marginal Abatement Cost (MAC)

In [14]:
from mac_pipeline import run_mac_analysis

mac_df = run_mac_analysis(
    df_decomposed           = df_decomposed,
    cb_data                 = cb_data,
    att_primary             = att_primary,
    att_strategy            = att_strategy,
    iso_code3               = cfg.ISO_CODE3,
    region                  = cfg.REGION,
    invent_dir              = cfg.INVENT_DIR,
    invent_historic_path    = cfg.INVENT_HISTORIC_PATH,   # invent_historic_mar.csv
    targets_path            = cfg.TARGETS_PATH,
    run_output_dir          = cfg.RUN_ID_OUTPUT_DIR,
    strategy_code_baseline  = cfg.STRATEGY_CODE_BASELINE,
    output_filename         = cfg.OUTPUT_MAC.name,
    year_cumul_start        = cfg.YEAR_CUMUL_START,       # None → 2018 (last inventory year)
    year_start              = cfg.YEAR_START,
)
logger.info(f"mac_df : {mac_df.shape}  → {cfg.OUTPUT_MAC}")


2026-09-17 17:24:46,284 - INFO - mac_df : (43, 9)  → /Users/fabianfuentes/git/ssp_morocco/ssp_modeling/ssp_run_output/sisepuede_results_sisepuede_run_2026-09-17T16;54;17.970763/marginal_abatement_costs_tornado.csv


## 5 · Tableau export

In [15]:
from mac_pipeline import build_attribute_map

att_map = build_attribute_map(
    att_primary    = att_primary,
    att_strategy   = att_strategy,
    run_output_dir = cfg.RUN_ID_OUTPUT_DIR,
)
logger.info(f"att_map : {att_map.shape}  → {cfg.RUN_ID_OUTPUT_DIR / 'ATTRIBUTE_MAP_TORNADO_WHIRLPOOL.csv'}")

2026-09-17 17:24:46,297 - INFO - att_map : (43, 8)  → /Users/fabianfuentes/git/ssp_morocco/ssp_modeling/ssp_run_output/sisepuede_results_sisepuede_run_2026-09-17T16;54;17.970763/ATTRIBUTE_MAP_TORNADO_WHIRLPOOL.csv


In [16]:
from mac_pipeline import build_tableau_tornado

# Exclude full-portfolio rows (e.g. PFLO:CONDITIONAL) — keep only individual transformations
mac_df = mac_df[mac_df["transformation_code"].notna() & (mac_df["transformation_code"] != "")]

tableau_tornado = build_tableau_tornado(
    mac_df         = mac_df,
    run_output_dir = cfg.RUN_ID_OUTPUT_DIR,
    tableau_dir    = cfg.TABLEAU_DIR,
    output_path    = cfg.OUTPUT_TABLEAU_TORNADO,
)
logger.info(f"tableau_tornado : {tableau_tornado.shape}  → {cfg.OUTPUT_TABLEAU_TORNADO}")

2026-09-17 17:24:46,311 - INFO - tableau_tornado : (42, 14)  → /Users/fabianfuentes/git/ssp_morocco/ssp_modeling/tableau/data/tableau_tornado.csv


## 6 · Summary

In [17]:
import pandas as pd

summary = pd.DataFrame([
    {"output": "decomposed SSP",     "rows": len(df_decomposed),   "cols": df_decomposed.shape[1],   "file": str(cfg.OUTPUT_DECOMPOSED)},
    {"output": "cost-benefits data", "rows": len(cb_data),         "cols": cb_data.shape[1],         "file": str(cfg.OUTPUT_CB_DATA)},
    {"output": "MAC curves",         "rows": len(mac_df),          "cols": mac_df.shape[1],          "file": str(cfg.OUTPUT_MAC)},
    {"output": "tableau tornado",    "rows": len(tableau_tornado), "cols": tableau_tornado.shape[1], "file": str(cfg.OUTPUT_TABLEAU_TORNADO)},
])
print(summary.to_string(index=False))

# Sanity checks on the tornado output
n_no_mac = tableau_tornado["marginal_abatement_cost"].isna().sum()
print(f"\nStrategies without a MAC (zero emission_diff): {n_no_mac}")
print("\nTop 10 by abatement (most negative emission_diff):")
print(
    tableau_tornado
    .nsmallest(10, "emission_diff")[
        ["transformation_name_sector", "emission_diff", "technical_cost", "marginal_abatement_cost"]
    ]
    .to_string(index=False)
)


            output   rows  cols                                                                                                                                                             file
    decomposed SSP   1452  4119    /Users/fabianfuentes/git/ssp_morocco/ssp_modeling/ssp_run_output/sisepuede_results_sisepuede_run_2026-09-17T16;54;17.970763/decomposed_ssp_output_tornado.csv
cost-benefits data 305978    21       /Users/fabianfuentes/git/ssp_morocco/ssp_modeling/ssp_run_output/sisepuede_results_sisepuede_run_2026-09-17T16;54;17.970763/cost_benefits_data_tornado.csv
        MAC curves     42     9 /Users/fabianfuentes/git/ssp_morocco/ssp_modeling/ssp_run_output/sisepuede_results_sisepuede_run_2026-09-17T16;54;17.970763/marginal_abatement_costs_tornado.csv
   tableau tornado     42    14                                                                               /Users/fabianfuentes/git/ssp_morocco/ssp_modeling/tableau/data/tableau_tornado.csv

Strategies without a MAC (zero emi